# 05 — Olist | Visualización ejecutiva de las 7 tablas Gold

**Dashboard en Databricks, sin escribir ni modificar datos.** Todos los gráficos se alimentan directamente de `workspace.olist_lakehouse.gold_*`. Elegí un mes en el widget superior y volvé a ejecutar las celdas para explorar ese período.

**Criterio de negocio:** ventas de pedidos *delivered*, atribuidas al mes de compra; GMV = precio de ítems sin flete. Los importes se expresan en **reales brasileños (BRL)**. Los meses con poco volumen deben interpretarse con cautela.

In [0]:
%pip install plotly
from pyspark.sql import functions as F
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display as ipy_display

BASE = "workspace.olist_lakehouse"
GOLD_TABLES = [
    "gold_monthly_sales",
    "gold_monthly_category_share",
    "gold_monthly_top_products",
    "gold_monthly_category_state",
    "gold_delivery_performance",
    "gold_monthly_payment_methods",
    "gold_monthly_portfolio_evolution"
]
for name in GOLD_TABLES:
    full = f"{BASE}.{name}"
    if not spark.catalog.tableExists(full):
        raise RuntimeError(f"No existe {full}. Ejecutá primero 03_build_gold_v2.")

meses = [r["mes"].strftime("%Y-%m") for r in spark.table(f"{BASE}.gold_monthly_sales")
         .select("mes").distinct().orderBy("mes").collect()]
if not meses:
    raise RuntimeError("Gold Sales está vacía.")
dbutils.widgets.dropdown("mes_foco", meses[-1], meses, "Mes para análisis detallado")
dbutils.widgets.dropdown("tabla_gold", GOLD_TABLES[0], GOLD_TABLES, "Tabla Gold para explorar")
MES = dbutils.widgets.get("mes_foco")
MES_FECHA = MES + "-01"
print("Mes seleccionado:", MES)

# Siete tablas agregadas, volumen moderado; Spark sigue siendo el origen de datos.
sales = spark.table(f"{BASE}.gold_monthly_sales").orderBy("mes").toPandas()
cat = spark.table(f"{BASE}.gold_monthly_category_share").toPandas()
top = spark.table(f"{BASE}.gold_monthly_top_products").toPandas()
states = spark.table(f"{BASE}.gold_monthly_category_state").toPandas()
delivery = spark.table(f"{BASE}.gold_delivery_performance").toPandas()
payments = spark.table(f"{BASE}.gold_monthly_payment_methods").toPandas()
portfolio = spark.table(f"{BASE}.gold_monthly_portfolio_evolution").orderBy("mes").toPandas()
for frame in [sales,cat,top,states,delivery,payments,portfolio]:
    frame["mes"] = pd.to_datetime(frame["mes"])

INK = "#15263D"
BLUE = "#2563EB"
TEAL = "#0D9488"
PURPLE = "#7C3AED"
AMBER = "#D97706"
RED = "#D9485F"
# Convertir columnas numéricas a tipos compatibles con Pandas y Plotly

columnas_numericas = [
    "gmv_brl",
    "flete_brl",
    "pagos_brl",
    "ticket_promedio_brl",
    "participacion_gmv_pct",
    "gmv_total_mes_brl",
    "tasa_entregas_tardias_pct",
    "hhi",
    "top5_share_pct",
    "top10_share_pct"
]

for frame in [sales, cat, top, states, delivery, payments, portfolio]:
    for columna in columnas_numericas:
        if columna in frame.columns:
            frame[columna] = pd.to_numeric(
                frame[columna],
                errors="coerce"
            )
def polish(fig, title, height=410, percent=False):
    fig.update_layout(
        template="plotly_white", title=dict(text=title, x=0, font=dict(size=20,color=INK)),
        height=height, font=dict(family="Arial", size=12,color=INK),
        margin=dict(l=30,r=20,t=70,b=30),
        paper_bgcolor="#FFFFFF",plot_bgcolor="#FFFFFF",
        hovermode="x unified" if fig.data and fig.data[0].type == "scatter" else "closest",
        legend=dict(orientation="h",yanchor="bottom",y=1.02,x=0)
    )
    fig.update_xaxes(showgrid=False, linecolor="#E2E8F0")
    fig.update_yaxes(gridcolor="#ECF0F5", zeroline=False)
    return fig

def fmt_brl(x):
    return "R$ " + f"{float(x):,.0f}".replace(",", ".")

def month_only(frame):
    return frame[frame["mes"].dt.strftime("%Y-%m") == MES].copy()

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Mes seleccionado: 2018-08


## Panorama general

**Qué muestra:** indicadores principales de ventas, actividad comercial y diversidad de categorías de todo el período disponible.

In [0]:
from html import escape
ms = month_only(sales)
ps = month_only(portfolio)
if ms.empty or ps.empty:
    raise RuntimeError(f"Faltan datos de {MES} en ventas o portfolio")
row_s, row_p = ms.iloc[0], ps.iloc[0]
indicators = [
    ("GMV mensual", fmt_brl(row_s["gmv_brl"]), "Sin flete"),
    ("Pedidos entregados", f"{int(row_s['pedidos_entregados']):,}".replace(",","."), "Mes de compra"),
    ("Ticket promedio", fmt_brl(row_s["ticket_promedio_brl"] or 0), "Por pedido con ítems"),
    ("Categorías activas", f"{int(row_p['categorias_activas'])}", "Excluye Sin categoría"),
]
css = "display:inline-block;width:21%;min-width:180px;vertical-align:top;margin:0 1% 10px 0;padding:20px;background:#F7FAFF;border:1px solid #E2E8F0;border-radius:16px;font-family:Arial;"
html = "<div>"+"".join(
    f'<div style="{css}"><div style="color:#6B7280;font-size:12px">{escape(t)}</div>'
    f'<div style="font-size:25px;font-weight:750;color:#15263D;margin:9px 0">{escape(v)}</div>'
    f'<div style="font-size:11px;color:#64748B">{escape(sub)}</div></div>'
    for t,v,sub in indicators
)+"</div>"
displayHTML(html)

GMV mensual R$ 838.577 Sin flete Pedidos entregados 6.351 Mes de compra Ticket promedio R$ 132 Por pedido con ítems Categorías activas 65 Excluye Sin categoría

## 1. Evolución mensual de ventas — `gold_monthly_sales`

**Qué muestra:** cómo cambia el GMV de artículos en BRL mes a mes. No incluye fletes, por lo que no representa el total cobrado.

In [0]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=sales["mes"], y=sales["gmv_brl"].astype(float), mode="lines+markers",
    line=dict(color=BLUE,width=3), marker=dict(size=7), fill="tozeroy", fillcolor="rgba(37,99,235,0.08)",
    name="GMV", hovertemplate="%{x|%b %Y}: R$ %{y:,.0f}<extra></extra>"))
polish(fig, "Ingresos comerciales por mes (GMV)")
fig.update_yaxes(title_text="BRL",tickprefix="R$ ",tickformat="~s")
fig.show()

<!doctype html>

## 2. Ticket promedio — `gold_monthly_sales`

**Qué muestra:** GMV dividido por la cantidad de pedidos entregados con al menos un ítem. Una subida no implica necesariamente más pedidos.

In [0]:
fig=go.Figure(go.Bar(x=sales["mes"],y=sales["ticket_promedio_brl"].astype(float),
    marker_color=TEAL, hovertemplate="%{x|%b %Y}: R$ %{y:,.2f}<extra></extra>"))
polish(fig,"Ticket promedio por pedido con ítems")
fig.update_yaxes(title_text="BRL por pedido",tickprefix="R$ ")
fig.show()

<!doctype html>

## 3. Categorías líderes en el mes — `gold_monthly_category_share`

**Qué muestra:** las categorías de mayor participación en el GMV del mes elegido. La cuota utiliza precios de ítems, sin flete.

In [0]:
cc = month_only(cat).sort_values("gmv_brl", ascending=False).head(12)
if cc.empty: print("Sin categorías para", MES)
else:
    cc=cc.sort_values("participacion_gmv_pct")
    fig=go.Figure(go.Bar(y=cc["categoria_analitica"],x=cc["participacion_gmv_pct"].astype(float),
        orientation="h", marker_color=PURPLE,
        customdata=cc["gmv_brl"].astype(float),
        hovertemplate="%{y}: %{x:.2f}% · R$ %{customdata:,.0f}<extra></extra>"))
    polish(fig, f"Top 12 categorías por participación · {MES}", height=490)
    fig.update_xaxes(title_text="% del GMV del mes",ticksuffix="%")
    fig.show()

<!doctype html>

## 4. Evolución de categorías — `gold_monthly_category_share`

**Qué muestra:** cómo varía la participación mensual de las ocho categorías que más GMV acumulan en el histórico.

In [0]:
cat8 = (cat.groupby("categoria_analitica",as_index=False)["gmv_brl"].sum()
        .nlargest(8,"gmv_brl")["categoria_analitica"].tolist())
small=cat[cat["categoria_analitica"].isin(cat8)].copy()
small["periodo"]=small["mes"].dt.strftime("%Y-%m")
pivot=small.pivot_table(index="categoria_analitica",columns="periodo",
    values="participacion_gmv_pct", aggfunc="sum").fillna(0)
fig=px.imshow(pivot, text_auto=".1f",aspect="auto",color_continuous_scale="Blues",
    labels=dict(color="% GMV",x="Mes",y="Categoría"))
polish(fig,"Participación por categoría a lo largo del tiempo",height=480)
fig.update_layout(coloraxis_colorbar=dict(title="%"))
fig.show()

<!doctype html>

## 5. Productos más vendidos — `gold_monthly_top_products`

**Qué muestra:** ranking mensual por unidades vendidas. Olist identifica los productos mediante `product_id`, no por un nombre comercial.

Lamentablemente los SKU's no tienen una descripción

In [0]:
tt=month_only(top).sort_values("ranking_mes")
if tt.empty:print("Sin productos rankeados en",MES)
else:
    tt["etiqueta"]=tt["product_id"].astype(str).str.slice(0,10)+"…"
    tt=tt.sort_values("unidades_vendidas")
    fig=go.Figure(go.Bar(x=tt["unidades_vendidas"],y=tt["etiqueta"],orientation="h",
        marker_color=BLUE,customdata=tt["product_id"],
        hovertemplate="ID %{customdata}<br>%{x} unidades<extra></extra>"))
    polish(fig, f"Top productos por unidades · {MES}",height=465)
    fig.update_xaxes(title_text="Unidades")
    fig.show()

<!doctype html>

## 6. Categorías y estados de destino — `gold_monthly_category_state`

**Qué muestra:** cruce del GMV de categorías en portugués y los estados de los clientes. El mapa de calor considera los ocho estados y categorías de mayor GMV del mes.

In [0]:
st=month_only(states)
if st.empty:print("Sin estados para",MES)
else:
    top_states=st.groupby("estado")["gmv_brl"].sum().nlargest(8).index
    top_categories=st.groupby("categoria_pt")["gmv_brl"].sum().nlargest(8).index
    matrix=(st[st["estado"].isin(top_states)&st["categoria_pt"].isin(top_categories)]
       .pivot_table(index="estado",columns="categoria_pt",values="gmv_brl",aggfunc="sum")
       .fillna(0).astype(float))
    fig=px.imshow(matrix,aspect="auto",color_continuous_scale="Teal",
       labels=dict(x="Categoría (PT)",y="Estado",color="GMV BRL"))
    polish(fig,f"Ventas por estado y categoría · {MES}",height=500)
    fig.update_xaxes(tickangle=-35)
    fig.show()

<!doctype html>

## 7. Puntualidad nacional — `gold_delivery_performance`

**Qué muestra:** porcentaje de entregas tardías respecto de las entregas con fechas evaluables a nivel Brasil. No incluye registros sin fechas suficientes.

In [0]:
br=delivery[delivery["nivel_geografico"]=="BRASIL"].sort_values("mes")
fig=go.Figure()
fig.add_trace(go.Scatter(x=br["mes"], y=br["porcentaje_tardias"].astype(float),
    mode="lines+markers",name="Entregas tardías",line=dict(color=RED,width=3),marker=dict(size=7)))
polish(fig,"Entregas tardías · Brasil")
fig.update_yaxes(title_text="% de entregas evaluables",ticksuffix="%",rangemode="tozero")
fig.show()

<!doctype html>

## 8. Retrasos por estado — `gold_delivery_performance`

**Qué muestra:** estados con mayor tasa de entregas tardías en el mes. Para reducir distorsiones se muestran solo estados con al menos 30 entregas evaluables.

In [0]:
dd=month_only(delivery)
dd=dd[(dd["nivel_geografico"]=="ESTADO") & (dd["entregas_evaluables"]>=30)]
dd=dd.sort_values("porcentaje_tardias",ascending=False).head(15).sort_values("porcentaje_tardias")
if dd.empty:print("Sin estados con 30 o más entregas evaluables en", MES)
else:
    fig=go.Figure(go.Bar(y=dd["estado"],x=dd["porcentaje_tardias"].astype(float),
        orientation="h",marker_color=AMBER,customdata=dd["entregas_evaluables"],
        hovertemplate="%{y}: %{x:.1f}% · %{customdata} evaluables<extra></extra>"))
    polish(fig,f"Estados con más entregas tardías · {MES}",height=460)
    fig.update_xaxes(title_text="% tardías",ticksuffix="%")
    fig.show()

<!doctype html>

## 9. Medios de pago — `gold_monthly_payment_methods`

**Qué muestra:** distribución del **importe cobrado** por método en el mes. No es participación de pedidos: una orden puede combinar métodos.

In [0]:
pp=month_only(payments)
if pp.empty:print("Sin pagos registrados para",MES)
else:
    fig=go.Figure(go.Pie(labels=pp["metodo_pago"], values=pp["valor_pagado_brl"].astype(float),
        hole=0.64, textinfo="label+percent", marker=dict(line=dict(color="#FFFFFF",width=3)),
        hovertemplate="%{label}: R$ %{value:,.0f} (%{percent})<extra></extra>"))
    polish(fig,f"Participación monetaria de medios de pago · {MES}",height=450)
    fig.update_layout(showlegend=True)
    fig.show()

<!doctype html>

## 10. Evolución del portfolio — `gold_monthly_portfolio_evolution`

**Qué muestra:** cantidad de categorías con ventas en cada mes, nuevas apariciones y reactivaciones luego de un período sin ventas.

In [0]:
p=portfolio.sort_values("mes")
fig=go.Figure()
for col,label,color in [
    ("categorias_activas","Activas",BLUE),
    ("categorias_nuevas","Nuevas",TEAL),
    ("categorias_reactivadas","Reactivadas",AMBER)]:
    fig.add_trace(go.Scatter(x=p["mes"],y=p[col],name=label,mode="lines+markers",
        line=dict(color=color,width=3 if col=="categorias_activas" else 2)))
polish(fig,"Cantidad de categorías activas, nuevas y reactivadas")
fig.update_yaxes(title_text="Categorías",rangemode="tozero")
fig.show()

<!doctype html>

## 11. Concentración del portfolio — `gold_monthly_portfolio_evolution`

**Qué muestra:** cuota del GMV aportada por las 5 y 10 categorías líderes. Valores más altos indican mayor dependencia de pocas categorías.

In [0]:
fig=go.Figure()
fig.add_trace(go.Scatter(x=p["mes"],y=p["participacion_top5_pct"],
    name="Top 5",line=dict(color=PURPLE,width=3),mode="lines+markers"))
fig.add_trace(go.Scatter(x=p["mes"],y=p["participacion_top10_pct"],
    name="Top 10",line=dict(color=TEAL,width=3),mode="lines+markers"))
polish(fig,"Concentración de ventas en las categorías líderes")
fig.update_yaxes(title_text="% del GMV",ticksuffix="%",range=[0,100])
fig.show()

<!doctype html>

## 12. Data Quality

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from decimal import Decimal

BASE = "workspace.olist_lakehouse"

# 1. Auditoría de las ejecuciones
audit = spark.table(f"{BASE}.pipeline_run_audit")

# 2. Auditoría de duplicados por batch
alerts = spark.table(f"{BASE}.silver_duplicate_alerts")

# 3. Última ejecución registrada
ultima = (
    audit.orderBy(F.col("recorded_at").desc())
    .limit(1)
    .collect()[0]
)

# 4. Último batch auditado por Silver
ultimo_batch = (
    alerts.orderBy(F.col("recorded_at").desc())
    .select("bronze_batch_id", "silver_run_id")
    .first()
)

batch_id = ultimo_batch["bronze_batch_id"] if ultimo_batch else None

# 5. Duplicados correspondientes a ese batch
duplicados = (
    alerts.filter(F.col("bronze_batch_id") == batch_id)
    .agg(
        F.sum("rows_received").alias("recibidos"),
        F.sum("exact_duplicate_rows").alias("duplicados"),
        F.sum("new_key_rows").alias("nuevos"),
        F.sum("conflicting_version_rows").alias("conflictos")
    )
    .first()
)

# 6. Calidad de pedidos Silver
orders = spark.table(f"{BASE}.silver_orders")
customers = spark.table(f"{BASE}.silver_customers")

pedidos_duplicados = (
    orders.count()
    - orders.select("order_id").distinct().count()
)

pedidos_sin_cliente = (
    orders.join(
        customers.select("customer_id"),
        "customer_id",
        "left_anti"
    ).count()
)

# 7. Estado de idempotencia
comparada = ultima["idempotence_compared"]
aprobada = ultima["idempotence_passed"]

if comparada and aprobada:
    estado_idempotencia = "PASS"
elif comparada and not aprobada:
    estado_idempotencia = "FAIL"
else:
    estado_idempotencia = "NO EVALUADA"

# 8. Construir resumen
resultado = [{
    "run_id": ultima["run_id"],
    "bronze_batch_id": batch_id,
    "bronze_rows": ultima["bronze_rows"],
    "silver_rows": ultima["silver_rows"],
    "gold_rows": ultima["gold_rows"],
    "duplicados_detectados": duplicados["duplicados"],
    "claves_nuevas": duplicados["nuevos"],
    "versiones_conflictivas": duplicados["conflictos"],
    "pedidos_duplicados_silver": pedidos_duplicados,
    "pedidos_sin_cliente": pedidos_sin_cliente,
    "idempotencia": estado_idempotencia,
    "controles_fallidos": ultima["checks_failed"],
    "gmv_brl": float(ultima["gmv_total_brl"]),
    "estado_pipeline": ultima["status"]
}]

# Orden secuencial de la auditoría Medallion

columnas_ordenadas = [
    # 1. Identificación de la ejecución
    "run_id",
    "bronze_batch_id",

    # 2. BRONZE - Ingesta
    "bronze_rows",

    # 3. SILVER - Depuración
    "duplicados_detectados",
    "claves_nuevas",
    "versiones_conflictivas",
    "silver_rows",

    # 4. GOLD - Resultados analíticos
    "gold_rows",
    "gmv_brl",

    # 5. Calidad de datos
    "pedidos_duplicados_silver",
    "pedidos_sin_cliente",

    # 6. Idempotencia
    "idempotencia",

    # 7. Resultado final
    "controles_fallidos",
    "estado_pipeline"
]

df_auditoria = spark.createDataFrame(resultado)

display(df_auditoria.select(*columnas_ordenadas))

run_id,bronze_batch_id,bronze_rows,duplicados_detectados,claves_nuevas,versiones_conflictivas,silver_rows,gold_rows,gmv_brl,pedidos_duplicados_silver,pedidos_sin_cliente,idempotencia,controles_fallidos,estado_pipeline
0e605f32-3f9c-496e-9b4c-aaaa89842467,1f7d850d-bb2c-49c3-957a-907c005350db,4652766,1550922,0,0,569774,13867,1.322149811E7,0,0,PASS,0,PASS
